# LlamaIndex components from documents to retrieval

An original, deterministic LlamaIndex companion lesson.

## Learning objectives

- Prepare documents and overlapping token nodes.
- Build a small local vector-like index abstraction.
- Retrieve the Ada Lovelace record with deterministic token similarity.

## Setup

The offline pipeline uses standard-library tokenization and set similarity. It deliberately avoids remote embeddings while preserving the document, node, index, and retriever boundaries.


The optional live example uses `python -m pip install llama-index llama-index-llms-huggingface-api`. Optional Phoenix tracing uses `python -m pip install arize-phoenix-otel`.

## Complete implementation

Run the next cell to build the complete offline demonstration.

In [1]:
import re

documents = [
    {
        "name": "Ada Lovelace",
        "text": "Ada Lovelace wrote notes on the Analytical Engine and described an algorithm.",
    },
    {
        "name": "Grace Hopper",
        "text": "Grace Hopper developed compiler technology and helped standardize COBOL.",
    },
    {
        "name": "Alan Turing",
        "text": "Alan Turing studied computation and codebreaking with abstract machines.",
    },
]

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.casefold())

def make_nodes(document, chunk_size=7, overlap=2):
    tokens = tokenize(document["text"])
    nodes = []
    start = 0
    while start < len(tokens):
        chunk = tokens[start : start + chunk_size]
        nodes.append(
            {
                "document_name": document["name"],
                "tokens": chunk,
                "text": " ".join(chunk),
            }
        )
        if start + chunk_size >= len(tokens):
            break
        start += chunk_size - overlap
    return nodes

prepared_nodes = [
    node for document in documents for node in make_nodes(document)
]

class LocalVectorLikeIndex:
    """A tiny sparse token-presence index with a retriever-like interface."""

    def __init__(self, nodes):
        self.nodes = list(nodes)
        self.vectors = [set(node["tokens"]) for node in self.nodes]

    def retrieve(self, query, top_k=1):
        query_vector = set(tokenize(query))
        scored = []
        for node, vector in zip(self.nodes, self.vectors):
            score = len(query_vector & vector) / max(1, len(query_vector | vector))
            scored.append((score, node["document_name"], node))
        ranked = sorted(scored, key=lambda item: (-item[0], item[1], item[2]["text"]))
        return [item[2] for item in ranked[:top_k]]

local_index = LocalVectorLikeIndex(prepared_nodes)
retrieved_nodes = local_index.retrieve("Analytical Engine algorithm", top_k=1)
retrieved_name = retrieved_nodes[0]["document_name"]

## Deterministic assertions

Run these checks to verify the lesson without credentials, network access, or third-party packages.

In [2]:
assert len(prepared_nodes) >= len(documents)
assert any(
    left["document_name"] == right["document_name"]
    and set(left["tokens"][-2:]) == set(right["tokens"][:2])
    for left, right in zip(prepared_nodes, prepared_nodes[1:])
)
assert retrieved_nodes[0]["text"] == "the analytical engine and described an algorithm"
assert retrieved_name == "Ada Lovelace"

print("Offline assertions passed.")

Offline assertions passed.


## Results interpretation

Documents retain record-level identity, nodes provide retrieval-sized passages, and the index owns query scoring. Token overlap is intentionally simple, but the interfaces mirror the separations used by production retrieval systems.

## Credentials

No credential or network is used offline. The optional query cell checks `HF_TOKEN` before LlamaIndex imports and checks `PHOENIX_API_KEY` before Phoenix imports. It never prints either value.

In [3]:
import os

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    print("SKIPPED: Set HF_TOKEN to run this optional LlamaIndex example.")
else:
    try:
        from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
        from llama_index.core import Document, SummaryIndex
    except ImportError:
        print("SKIPPED: llama-index is not installed; run `python -m pip install llama-index llama-index-llms-huggingface-api`.")
    else:
        phoenix_api_key = os.getenv("PHOENIX_API_KEY")
        if phoenix_api_key:
            try:
                from phoenix.otel import register
            except ImportError:
                print("SKIPPED: Phoenix tracing is not installed; run `python -m pip install arize-phoenix-otel`.")
            else:
                tracer_provider = register(
                    project_name="hf-agents-course-llama-index",
                    endpoint="https://app.phoenix.arize.com/v1/traces",
                    headers={"api_key": phoenix_api_key},
                    auto_instrument=True,
                )
                print("Phoenix tracing configured.")
        else:
            print("SKIPPED: PHOENIX_API_KEY is absent; tracing remains disabled.")

        llm = HuggingFaceInferenceAPI(
            model_name="Qwen/Qwen2.5-7B-Instruct",
            token=hf_token,
            task="text-generation",
        )
        live_documents = [
            Document(text="Ada Lovelace wrote notes about the Analytical Engine."),
            Document(text="Grace Hopper developed compiler technology."),
        ]
        live_index = SummaryIndex.from_documents(live_documents)
        live_query_engine = live_index.as_query_engine(llm=llm)
        print("Configured a live SummaryIndex query engine; querying sends content to the hosted model.")

SKIPPED: PHOENIX_API_KEY is absent; tracing remains disabled.


Configured a live SummaryIndex query engine; querying sends content to the hosted model.


## Completion checklist

- [ ] Prepared overlapping nodes from the documents.
- [ ] Inspected the local vector-like index.
- [ ] Verified the Ada Lovelace retrieval result.

## Attribution

This compact retrieval implementation is original. Its component boundaries are informed by the official [LlamaIndex loading guide](https://docs.llamaindex.ai/en/stable/module_guides/loading/) and [querying guide](https://docs.llamaindex.ai/en/stable/module_guides/querying/); no official example is reproduced. Notebook code carries the Apache-2.0 metadata license.